# Evaluation

In [3]:
from pydantic import BaseModel
import json

def read_json(file_path : str) -> dict:
    with open(file_path, 'r') as f:
        return json.load(f)

In [4]:
import os

goal_settings = {}
analysises = {}

for dir in os.listdir('./results'):
    file_path = os.path.join('./results', dir, 'goal_setting.json')
    if os.path.exists(file_path):
        analysises[dir] = read_json(os.path.join('./results', dir, 'analysis.json'))
        goal_settings[dir] = read_json(file_path)

## G-Eval

### Сравнение сгенерированных и СЭР ЛО

In [5]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

CRITERIA = """
Оцени соответствие стратегии муниципального образования стратегическим целям региона (уровень выше).

Игнорируй:
- структуру документа
- наличие/отсутствие конкретных показателей
- объём и детализацию

Оценивай ТОЛЬКО смысловое соответствие:
1. Являются ли цели МО логическим продолжением/декомпозицией целей ЛО?
2. Можно ли достижение целей ЛО представить без целей МО?
3. Есть ли явные противоречия?
"""

# Создаём G-Eval метрику для оценки ответа
geval_metric = GEval(
    name="StrategyQuality",
    criteria=CRITERIA,
    evaluation_params=[
        LLMTestCaseParams.EXPECTED_OUTPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT
    ],
    evaluation_steps= [
        "Прочитай стратегические цели региона (Expected Output)",
        "Прочитай миссию и цели муниципального образования (Actual Output)",
        "Для каждой цели региона определи, отражена ли она в МО",
        "Поставь итоговую оценку и объясни почему"
    ],
    threshold=0.5,
    model="openai/gpt-4.1",
)

In [6]:
test_cases = []

def read_txt(file_path : str) -> str:
    with open(file_path, 'r', encoding='utf-8') as f:
        return f.read().strip()

for dir in os.listdir('./results'):
    if '.' in dir:
        continue
    expected_output = read_txt(os.path.join('./results', 'expected.txt'))
    actual_output = read_json(os.path.join('./results', dir, 'goal_setting.json'))
    # actual_output = read_txt(os.path.join('./results', dir, 'actual.txt'))
    test_case = LLMTestCase(
        name=dir,
        input=dir,
        actual_output=str(actual_output),
        expected_output=str(expected_output)
    )
    test_cases.append(test_case)

In [7]:
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig, DisplayConfig, ErrorConfig

async_config = AsyncConfig(
    run_async=True,        # включает параллельное выполнение
    max_concurrent=10      # максимальное количество параллельных тест-кейсов
)

# Настройка отображения результатов
display_config = DisplayConfig(
    print_results=False,    # печатает результаты каждого теста
    show_indicator=True,   # показывает прогресс-бар
    verbose_mode=False,    # отключает подробный вывод метрик (можете включить при отладке)
)

error_config = ErrorConfig(
    ignore_errors=False,   # не игнорировать ошибки (при True продолжит выполнение)
    skip_on_missing_params=False  # не пропускать тесты с отсутствующими параметрами
)

results = evaluate(
    test_cases=test_cases,
    metrics=[geval_metric],
    async_config=async_config,
    display_config=display_config,
    error_config=error_config
)

✨ You're running DeepEval's latest StrategyQuality [GEval] Metric! (using openai/gpt-4.1, strict=False, 
async_mode=True)...

Output()

⚠ WARNING: No hyperparameters logged.
» ]8;id=321043;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 50.24s | token cost: None)
» Test Results (18 total tests):
   » Pass Rate: 94.44% | Passed: 17 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [8]:
import numpy
numpy.mean([t.metrics_data[0].score for t in results.test_results])

np.float64(0.7666666666666667)

In [9]:
results.test_results

[TestResult(name='kirovsk', success=True, metrics_data=[MetricData(name='StrategyQuality [GEval]', threshold=0.5, success=True, score=0.9, reason='Стратегические цели региона охватывают демографию, здоровье, спорт, экспорт, продовольственную безопасность, транспорт, комфортную среду и туризм. В миссии и целях Кировского района отражены: демография (рост населения, продолжительность жизни), здоровье (доступность здравоохранения), спорт (объекты физкультуры), комфортная среда (благоустройство, инфраструктура), транспорт (развитие дорог, логистика), экономика (производительность, инвестиции). Продовольственная безопасность и туризм явно не выделены как самостоятельные цели, но частично могут быть косвенно затронуты через экономику и инфраструктуру. В целом, большинство региональных целей интегрированы, но есть незначительные пробелы по отдельным направлениям.', strict_mode=False, evaluation_model='openai/gpt-4.1', error=None, evaluation_cost=None, verbose_logs='Criteria:\n\nОцени соответс

### Сравнение существующих и СЭР ЛО

In [10]:
test_cases = []

def read_txt(file_path : str) -> str:
    with open(file_path, 'r', encoding='utf-8') as f:
        return f.read().strip()

for dir in os.listdir('./results'):
    if '.' in dir:
        continue
    expected_output = read_txt(os.path.join('./results', 'expected.txt'))
    # actual_output = read_json(os.path.join('./results', dir, 'goal_setting.json'))
    actual_output = read_txt(os.path.join('./results', dir, 'actual.txt'))
    test_case = LLMTestCase(
        name=dir,
        input=dir,
        actual_output=str(actual_output),
        expected_output=str(expected_output)
    )
    test_cases.append(test_case)

In [11]:
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig, DisplayConfig, ErrorConfig

async_config = AsyncConfig(
    run_async=True,        # включает параллельное выполнение
    max_concurrent=10      # максимальное количество параллельных тест-кейсов
)

# Настройка отображения результатов
display_config = DisplayConfig(
    print_results=False,    # печатает результаты каждого теста
    show_indicator=True,   # показывает прогресс-бар
    verbose_mode=False,    # отключает подробный вывод метрик (можете включить при отладке)
)

error_config = ErrorConfig(
    ignore_errors=False,   # не игнорировать ошибки (при True продолжит выполнение)
    skip_on_missing_params=False  # не пропускать тесты с отсутствующими параметрами
)

results = evaluate(
    test_cases=test_cases,
    metrics=[geval_metric],
    async_config=async_config,
    display_config=display_config,
    error_config=error_config
)

✨ You're running DeepEval's latest StrategyQuality [GEval] Metric! (using openai/gpt-4.1, strict=False, 
async_mode=True)...

Output()

⚠ WARNING: No hyperparameters logged.
» ]8;id=438310;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 12.49s | token cost: None)
» Test Results (18 total tests):
   » Pass Rate: 83.33% | Passed: 15 | Failed: 3

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [12]:
import numpy
numpy.mean([t.metrics_data[0].score for t in results.test_results])

np.float64(0.65)

In [14]:
results.test_results

[TestResult(name='kirovsk', success=False, metrics_data=[MetricData(name='StrategyQuality [GEval]', threshold=0.5, success=False, score=0.4, reason='В стратегических целях региона обозначены направления: демография, здоровье, спорт, экспорт, продовольственная безопасность, транспорт, комфортная среда, туризм. В целях МО отражены промышленность, предпринимательство, образование, ЖКХ, инвестиции, муниципальное управление. Частично отражены цели по промышленности (экспорт, развитие экономики), ЖКХ (комфортная среда), профориентации (долголетие), но отсутствуют прямые упоминания о демографии, здоровье, спорте, транспорте, туризме и продбезопасности. Итоговая оценка снижена за неполное соответствие ключевым региональным приоритетам.', strict_mode=False, evaluation_model='openai/gpt-4.1', error=None, evaluation_cost=None, verbose_logs='Criteria:\n\nОцени соответствие стратегии муниципального образования стратегическим целям региона (уровень выше).\n\nИгнорируй:\n- структуру документа\n- нали

## Contextual

In [10]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import FaithfulnessMetric, HallucinationMetric

# Инициализация метрик через ваш RouterAI
faithfulness_metric = FaithfulnessMetric(
    threshold=0.5,
    model="openai/gpt-4o-mini",
    include_reason=True
)

hallucination_metric = HallucinationMetric(
    threshold=0.5,
    model="openai/gpt-4o-mini",
    include_reason=True
)

test_cases = []

for dir in os.listdir('./results'):
    context = read_json(os.path.join('./results', dir, 'analysis.json'))
    expected_output = read_json(os.path.join('./results', dir, 'goal_setting.json'))
    test_case = LLMTestCase(
        name=dir,
        input=dir,
        actual_output=str(expected_output),
        expected_output=str(expected_output),
        context=[str(context)],
        retrieval_context=[str(context)]
    )
    test_cases.append(test_case)

In [ ]:
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig, DisplayConfig, ErrorConfig

async_config = AsyncConfig(
    run_async=True,        # включает параллельное выполнение
    max_concurrent=10      # максимальное количество параллельных тест-кейсов
)

# Настройка отображения результатов
display_config = DisplayConfig(
    print_results=True,    # печатает результаты каждого теста
    show_indicator=True,   # показывает прогресс-бар
    verbose_mode=False,    # отключает подробный вывод метрик (можете включить при отладке)
)

error_config = ErrorConfig(
    ignore_errors=False,   # не игнорировать ошибки (при True продолжит выполнение)
    skip_on_missing_params=False  # не пропускать тесты с отсутствующими параметрами
)

results = evaluate(
    test_cases=test_cases,
    metrics=[faithfulness_metric, hallucination_metric],
    async_config=async_config,
    display_config=display_config,
    error_config=error_config
)

✨ You're running DeepEval's latest Faithfulness Metric! (using openai/gpt-4o-mini, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Hallucination Metric! (using openai/gpt-4o-mini, strict=False, 
async_mode=True)...

Output()

In [ ]:
results

In [94]:
# from deepeval.test_case import LLMTestCase
# from deepeval.metrics import FaithfulnessMetric, HallucinationMetric
# from tqdm import tqdm
# import pandas as pd

# # Инициализация метрик через ваш RouterAI
# faithfulness_metric = FaithfulnessMetric(
#     threshold=0.5,
#     model="openai/gpt-4o-mini",
#     include_reason=True
# )

# hallucination_metric = HallucinationMetric(
#     threshold=0.5,
#     model="openai/gpt-4o-mini",
#     include_reason=True
# )

# results = []

# for key in tqdm(analysises.keys(), desc="Оценка галлюцинаций"):
#     # Берем контекст и ответ по одному ключу
#     context = []
#     for k,d in analysises[key].items():
#         for k_,v in d.items():
#             context.append(v)
#     response = goal_settings[key]
    
#     # Создаем тестовый кейс
#     test_case = LLMTestCase(
#         input=key,  # или сам запрос, если он у вас есть отдельно
#         actual_output=str(response),
#         retrieval_context=context,  # для Faithfulness
#         context=context              # для Hallucination
#     )
    
#     # Запускаем обе метрики
#     faithfulness_metric.measure(test_case)
#     hallucination_metric.measure(test_case)
    
#     # Сохраняем результаты
#     results.append({
#         "query_key": key,
#         "response": response[:200] + "..." if len(response) > 200 else response,
#         "context_length": len(context),
#         "faithfulness_score": faithfulness_metric.score,
#         "faithfulness_passed": faithfulness_metric.is_successful(),
#         "faithfulness_reason": faithfulness_metric.reason,
#         "hallucination_score": hallucination_metric.score,
#         "hallucination_passed": hallucination_metric.is_successful(),
#         "hallucination_reason": hallucination_metric.reason
#     })

# # Превращаем в DataFrame для анализа
# df_results = pd.DataFrame(results)

# # Выводим статистику
# print(f"\n=== СТАТИСТИКА ===")
# print(f"Всего оценено: {len(df_results)}")
# print(f"\nFaithfulness:")
# print(f"  Средний score: {df_results['faithfulness_score'].mean():.3f}")
# print(f"  Прошли порог: {df_results['faithfulness_passed'].sum()}/{len(df_results)} ({df_results['faithfulness_passed'].mean()*100:.1f}%)")
# print(f"\nHallucination:")
# print(f"  Средний score: {df_results['hallucination_score'].mean():.3f}")
# print(f"  Прошли порог: {df_results['hallucination_passed'].sum()}/{len(df_results)} ({df_results['hallucination_passed'].mean()*100:.1f}%)")

# # Показываем проблемные кейсы
# print(f"\n=== ПРОБЛЕМНЫЕ КЕЙСЫ (hallucination > 0.5) ===")
# problematic = df_results[df_results['hallucination_score'] > 0.5]
# for _, row in problematic.iterrows():
#     print(f"  {row['query_key']}: hallucination={row['hallucination_score']:.2f}")